# Step 2 - Process daily files from mike
Input: data from step 1 concatenated into one file
Check if all trajectories include 180 days of data
Add presence check and add outside area
Export
Output: 'processed_trajectories_wpresence_unfiltered.csv


In [1]:
import xarray as xr
import pandas as pd
from datetime import datetime, timedelta
import os

from shapely.geometry import Point, Polygon as ShapelyPolygon
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# File Paths to data
repo_path = '/Users/zephyrsylvester/repos/circumpolar-connectivity-analysis/data/'
output_path = '/Users/zephyrsylvester/repos/circumpolar-connectivity-analysis/processed_data/'

# List of simulations and locations
sim_list = ['007', '008', '009', '010', '011', '012', '013', '014', '015', '016', '017', '018']
locations = ['BS', 'GERL', 'GP', 'MB2']


In [3]:
# List all matching CSV files in the directory
csv_files = [os.path.join(output_path, f) for f in os.listdir(output_path) if f.startswith('proc_traj_') and f.endswith('.csv')]

# Read each CSV file into a DataFrame and concatenate them into one DataFrame
dataframes = [pd.read_csv(file) for file in csv_files]
concatenated_df = pd.concat(dataframes, ignore_index=True)
concatenated_df

,N,T,date,lat,lon,release,stage,larval_id,start_year,hypothesis,IDL_loc
0,0,0,2016-11-01,-66.912544,287.83374,0,0,00_16_1_0001,2016,h_null,BS
1,0,1,2016-11-02,-66.830450,288.03165,0,0,00_16_1_0001,2016,h_null,BS
2,0,2,2016-11-03,-66.757600,288.15454,0,0,00_16_1_0001,2016,h_null,BS
3,0,3,2016-11-04,-66.711136,288.16013,0,0,00_16_1_0001,2016,h_null,BS
4,0,4,2016-11-05,-66.718590,288.06876,0,0,00_16_1_0001,2016,h_null,BS
...,...,...,...,...,...,...,...,...,...,...,...
13994275,199,175,2019-09-12,-60.327457,306.02032,10,4,00_18_1_0200,2018,h_null,BS
13994276,199,176,2019-09-13,-60.238560,305.90730,10,4,00_18_1_0200,2018,h_null,BS
13994277,199,177,2019-09-14,-60.116886,305.83090,10,4,00_18_1_0200,2018,h_null,BS
13994278,199,178,2019-09-15,-60.023293,305.82330,10,4,00_18_1_0200,2018,h_null,BS


# Quality Check Data
1) do all larvae have 180 day trajectories?
2) do they all ACTUALLY pass through the given location?

In [4]:
def check_drifter_days(df):
    counts = df.groupby(['larval_id']).size()
    incorrect = counts[counts != 180]
    if not incorrect.empty:
        print("Drifters with incorrect number of days:")
        print(incorrect)
    else:
        print(f"All {df.larval_id.nunique()} larvae have 180 days of data.")

check_drifter_days(concatenated_df)

All 77746 larvae have 180 days of data.


In [5]:
# Define region coordinates
boxes = {
    'BS': [{'lonpoly': [298.0, 300.0, 300.0, 298.0], 'latpoly': [-63.7, -63.7, -62.6, -62.6]},
           {'lonpoly': [300.0, 302.0, 302.0, 300.0], 'latpoly': [-63.5, -63.5, -62.5, -62.5]}],
    'GERL': {'lonpoly': [296.1, 298.5, 299.6, 297.6], 'latpoly': [-64.7, -63.6, -64.0, -65.3]},
    'GP': {'lonpoly': [291.6, 295.5, 297.6, 294.0], 'latpoly': [-66.7, -64.5, -65.3, -67.4]},
    'MB2': {'lonpoly': [287.7, 290.5, 294.0, 292.0], 'latpoly': [-69.1, -67.5, -68.5, -69.7]}
}

In [6]:
# Function to apply presence check for all regions
def apply_presence_check_all(df):
    df = df.copy()
    for region, coords in boxes.items():
        if region == 'BS':
            polygons = [ShapelyPolygon(zip(coord['lonpoly'], coord['latpoly'])) for coord in coords]
        else:
            polygons = [ShapelyPolygon(zip(coords['lonpoly'], coords['latpoly']))]
        
        column_name = f'{region}'
        df[column_name] = df.apply(lambda row: any(polygon.contains(Point(row['lon'], row['lat'])) for polygon in polygons), axis=1)
    
    return df

In [7]:
%%time
# Apply the presence check function to the DataFrame
processed_df = apply_presence_check_all(concatenated_df)
processed_df

CPU times: user 11min 51s, sys: 19 s, total: 12min 10s
Wall time: 12min 19s


,N,T,date,lat,lon,release,stage,larval_id,start_year,hypothesis,IDL_loc,BS,GERL,GP,MB2
0,0,0,2016-11-01,-66.912544,287.83374,0,0,00_16_1_0001,2016,h_null,BS,False,False,False,False
1,0,1,2016-11-02,-66.830450,288.03165,0,0,00_16_1_0001,2016,h_null,BS,False,False,False,False
2,0,2,2016-11-03,-66.757600,288.15454,0,0,00_16_1_0001,2016,h_null,BS,False,False,False,False
3,0,3,2016-11-04,-66.711136,288.16013,0,0,00_16_1_0001,2016,h_null,BS,False,False,False,False
4,0,4,2016-11-05,-66.718590,288.06876,0,0,00_16_1_0001,2016,h_null,BS,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13994275,199,175,2019-09-12,-60.327457,306.02032,10,4,00_18_1_0200,2018,h_null,BS,False,False,False,False
13994276,199,176,2019-09-13,-60.238560,305.90730,10,4,00_18_1_0200,2018,h_null,BS,False,False,False,False
13994277,199,177,2019-09-14,-60.116886,305.83090,10,4,00_18_1_0200,2018,h_null,BS,False,False,False,False
13994278,199,178,2019-09-15,-60.023293,305.82330,10,4,00_18_1_0200,2018,h_null,BS,False,False,False,False


In [8]:
check_drifter_days(processed_df)

All 77746 larvae have 180 days of data.


In [9]:
# Define the presence columns
presence_columns = ['BS', 'GERL', 'GP', 'MB2']

# Convert 'Date' column to datetime format
processed_df = processed_df.copy()
processed_df['date'] = pd.to_datetime(processed_df['date'])
# Add 'outside' column to indicate if all presence columns are False
processed_df['outside'] = (processed_df[presence_columns] == 0).all(axis=1)
print('number of larvae:', processed_df.larval_id.nunique())

processed_df

number of larvae: 77746


,N,T,date,lat,lon,release,stage,larval_id,start_year,hypothesis,IDL_loc,BS,GERL,GP,MB2,outside
0,0,0,2016-11-01,-66.912544,287.83374,0,0,00_16_1_0001,2016,h_null,BS,False,False,False,False,True
1,0,1,2016-11-02,-66.830450,288.03165,0,0,00_16_1_0001,2016,h_null,BS,False,False,False,False,True
2,0,2,2016-11-03,-66.757600,288.15454,0,0,00_16_1_0001,2016,h_null,BS,False,False,False,False,True
3,0,3,2016-11-04,-66.711136,288.16013,0,0,00_16_1_0001,2016,h_null,BS,False,False,False,False,True
4,0,4,2016-11-05,-66.718590,288.06876,0,0,00_16_1_0001,2016,h_null,BS,False,False,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13994275,199,175,2019-09-12,-60.327457,306.02032,10,4,00_18_1_0200,2018,h_null,BS,False,False,False,False,True
13994276,199,176,2019-09-13,-60.238560,305.90730,10,4,00_18_1_0200,2018,h_null,BS,False,False,False,False,True
13994277,199,177,2019-09-14,-60.116886,305.83090,10,4,00_18_1_0200,2018,h_null,BS,False,False,False,False,True
13994278,199,178,2019-09-15,-60.023293,305.82330,10,4,00_18_1_0200,2018,h_null,BS,False,False,False,False,True


In [10]:
processed_df

,N,T,date,lat,lon,release,stage,larval_id,start_year,hypothesis,IDL_loc,BS,GERL,GP,MB2,outside
0,0,0,2016-11-01,-66.912544,287.83374,0,0,00_16_1_0001,2016,h_null,BS,False,False,False,False,True
1,0,1,2016-11-02,-66.830450,288.03165,0,0,00_16_1_0001,2016,h_null,BS,False,False,False,False,True
2,0,2,2016-11-03,-66.757600,288.15454,0,0,00_16_1_0001,2016,h_null,BS,False,False,False,False,True
3,0,3,2016-11-04,-66.711136,288.16013,0,0,00_16_1_0001,2016,h_null,BS,False,False,False,False,True
4,0,4,2016-11-05,-66.718590,288.06876,0,0,00_16_1_0001,2016,h_null,BS,False,False,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13994275,199,175,2019-09-12,-60.327457,306.02032,10,4,00_18_1_0200,2018,h_null,BS,False,False,False,False,True
13994276,199,176,2019-09-13,-60.238560,305.90730,10,4,00_18_1_0200,2018,h_null,BS,False,False,False,False,True
13994277,199,177,2019-09-14,-60.116886,305.83090,10,4,00_18_1_0200,2018,h_null,BS,False,False,False,False,True
13994278,199,178,2019-09-15,-60.023293,305.82330,10,4,00_18_1_0200,2018,h_null,BS,False,False,False,False,True


# Export

In [10]:
# Export
output_file = os.path.join(output_path, 'processed_trajectories_wpresence_unfiltered.csv')
processed_df.to_csv(output_file, index=False)

In [11]:
print('done')

done
